# Setup e deputados - Câmara/Brasil

Primeira etapa da pipeline nacional.

Cria as 27 pastas de UF dentro do Volume e baixa a lista da 57ª legislatura uma vez. O `deputados.csv` continua preservando o retorno da fonte; o resumo passa a separar quantidade de registros e quantidade de IDs únicos.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time

import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

CATALOGO = "workspace"
SCHEMA = "pi_ii_bronze"
VOLUME = "camara"
ID_LEGISLATURA = 57

UFS = [
    "AC", "AL", "AP", "AM", "BA", "CE", "DF", "ES", "GO",
    "MA", "MT", "MS", "MG", "PA", "PB", "PR", "PE", "PI",
    "RJ", "RN", "RS", "RO", "RR", "SC", "SP", "SE", "TO",
]

ROOT = Path(f"/Volumes/{CATALOGO}/{SCHEMA}/{VOLUME}")

print("Destino:", ROOT)


In [ ]:
# estrutura da bronze
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOGO}.{SCHEMA}.{VOLUME}")

for uf in UFS:
    pasta = ROOT / uf.lower()
    pasta.mkdir(parents=True, exist_ok=True)

    teste = pasta / "_teste_escrita.tmp"
    teste.write_text("ok", encoding="utf-8")
    teste.unlink()

print("Pastas de UF prontas:", len(UFS))


In [ ]:
# sessão HTTP com retry
BASE_URL = "https://dadosabertos.camara.leg.br/api/v2"

retry = Retry(
    total=6,
    connect=6,
    read=6,
    status=6,
    backoff_factor=1.0,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=frozenset(["GET"]),
    respect_retry_after_header=True,
)

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "User-Agent": "PI-II-Univesp-Bronze-Camara-Brasil/1.0",
})
session.mount("https://", HTTPAdapter(max_retries=retry))
session.mount("http://", HTTPAdapter(max_retries=retry))

def api_get(url, params=None):
    r = session.get(url, params=params, timeout=(30, 120))
    r.raise_for_status()
    return r.json()


In [ ]:
# deputados da 57ª legislatura
url = f"{BASE_URL}/deputados"
params = {
    "idLegislatura": ID_LEGISLATURA,
    "ordem": "ASC",
    "ordenarPor": "nome",
    "itens": 100,
}

registros = []
proxima_url = url
proxima_params = params

while proxima_url:
    payload = api_get(proxima_url, proxima_params)
    proxima_params = None

    dados = payload.get("dados", [])
    if isinstance(dados, dict):
        dados = [dados]

    registros.extend(dados)

    proxima_url = None
    for link in payload.get("links", []) or []:
        if str(link.get("rel", "")).lower() == "next":
            proxima_url = link.get("href")
            break

deputados = pd.json_normalize(registros, sep=".")

if deputados.empty:
    raise RuntimeError("Nenhum deputado foi retornado pela API.")

if "siglaUf" not in deputados.columns:
    raise RuntimeError(
        f"Coluna siglaUf não encontrada. Colunas: {list(deputados.columns)}"
    )

deputados["siglaUf"] = deputados["siglaUf"].astype(str).str.upper().str.strip()

print("Deputados retornados:", len(deputados))
print("UFs encontradas:", sorted(deputados["siglaUf"].unique()))


In [ ]:
# salva um deputados.csv em cada pasta de estado
id_col = "id" if "id" in deputados.columns else None
if not id_col:
    raise RuntimeError(
        f"Coluna id não encontrada em deputados. Colunas: {list(deputados.columns)}"
    )

resumo = []

for uf in UFS:
    recorte = deputados[deputados["siglaUf"].eq(uf)].copy()
    destino = ROOT / uf.lower() / "deputados.csv"

    recorte.to_csv(
        destino,
        sep=";",
        index=False,
        encoding="utf-8",
    )

    resumo.append({
        "uf": uf,
        "registros": len(recorte),
        "deputados_unicos": recorte[id_col].astype(str).nunique(),
    })

resumo_df = pd.DataFrame(resumo)

print("Registros no retorno:", int(resumo_df["registros"].sum()))
print("IDs únicos por UF (soma):", int(resumo_df["deputados_unicos"].sum()))

if "display" in globals():
    display(resumo_df)
else:
    print(resumo_df.to_string(index=False))


In [ ]:
# config da pipeline nacional
config = {
    "catalogo": CATALOGO,
    "schema": SCHEMA,
    "volume": VOLUME,
    "raiz": str(ROOT),
    "legislatura": ID_LEGISLATURA,
    "anos": [2023, 2024, 2025, 2026],
    "ufs": UFS,
    "gerado_em_utc": datetime.now(timezone.utc).isoformat(),
}

with (ROOT / "_config_brasil.json").open("w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

resumo_df.to_csv(
    ROOT / "_deputados_por_uf.csv",
    sep=";",
    index=False,
    encoding="utf-8",
)

print("Setup concluído.")
